# HydrAI — Cantera vs PINN vs SR Comparison

Validate the Main_7 PINN surrogate and the Main_8 symbolic-regression (SR) distillation against Cantera ground truth, on the same full axial profiles.

**Pipeline:** Main_7 (PINN) trained → Main_8 (SR) distils it → **Main_9 (this notebook)** compares both against Cantera → Main_10 optimises using the validated SR surrogate.

Prerequisites: run Main_7 with `IF_MODEL_EXPORT=True`, and Main_8 with `TEACHER_STEM='pinn_pfr'` and `IF_SR_EXPORT=True`.

Sections: 1 Setup · 2 Paths & flags · 3 Load PINN · 4 Load SR equations · 5 Load data & select runs · 6 Compute predictions · 7 Axial overlays · 8 Parity plots · 9 Metrics table · 10 Inference speed · 11 Export

In [ ]:
# ═════════════════════════════════════════════════════════════════════════
# 1. SETUP & IMPORTS
# ═════════════════════════════════════════════════════════════════════════
import os
import sys
import json
import time
import importlib.util
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import torch
from sklearn.metrics import r2_score, mean_absolute_error

project_root = Path(os.getcwd())
while not (project_root / 'src').exists() and project_root != project_root.parent:
    project_root = project_root.parent
sys.path.insert(0, str(project_root))
os.chdir(project_root)

from src.models import PINNPFR
from src.utils.plot_style import setup_matplotlib
from src.utils.run_log import start_run_log
from src.ml.dataframe_pickle import load_portable_pickle

setup_matplotlib()
start_run_log('Main_9_compare_cantera_pinn_sr')
print("Libraries imported successfully.")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════
# 2. PATHS & FLAGS
# ═════════════════════════════════════════════════════════════════════════
# Edit configs/ml/main9_compare_cantera_pinn_sr_config.json to change these
# (edit and re-run this cell; no kernel restart needed).
CONFIG_PATH = project_root / 'configs' / 'ml' / 'main9_compare_cantera_pinn_sr_config.json'
if CONFIG_PATH.exists():
    with open(CONFIG_PATH) as f:
        _cfg = json.load(f)
else:
    _cfg = {}
    print(f'[WARN] Config not found: {CONFIG_PATH}. Using defaults.')

IF_PLOT_SHOWN     = _cfg.get('if_plot_shown', True)
IF_PLOT_EXPORT    = _cfg.get('if_plot_export', True)
IF_EXPORT_REPORT  = _cfg.get('if_export_report', True)
SR_TEACHER_STEM   = _cfg.get('sr_teacher_stem', 'pinn_pfr')  # which Main_8 SR export to load
N_COMPARISON_RUNS = _cfg.get('n_comparison_runs', 6)         # runs sampled for axial overlay plots
RANDOM_STATE      = _cfg.get('random_state', 42)

_stem_to_sr_subdir = {'simple_nn_full_profile': 'sr_full_profile', 'pinn_pfr': 'sr_pinn'}
SR_SUBDIR = _stem_to_sr_subdir.get(SR_TEACHER_STEM, 'sr_custom')

MODELS_DIR = project_root / 'models'
SR_DIR     = MODELS_DIR / SR_SUBDIR
FIGS_DIR   = project_root / 'outputs' / 'figures' / 'Main_9_compare_cantera_pinn_sr'
FIGS_DIR.mkdir(parents=True, exist_ok=True)

print(f'SR teacher: {SR_TEACHER_STEM}  |  SR dir: {SR_DIR}')
print(f'Comparison runs: {N_COMPARISON_RUNS}  |  Random state: {RANDOM_STATE}')

In [ ]:
# ═════════════════════════════════════════════════════════════════════════
# 3. LOAD PINN MODEL (Main_7)
# ═════════════════════════════════════════════════════════════════════════
pinn_manifest_path = MODELS_DIR / 'pinn_pfr' / 'pinn_pfr_manifest.json'
if not pinn_manifest_path.exists():
    raise FileNotFoundError(
        f'{pinn_manifest_path} not found — run Main_7 with IF_MODEL_EXPORT=True first.'
    )

with open(pinn_manifest_path) as f:
    pinn_manifest = json.load(f)

arch              = pinn_manifest['architecture']
feature_cols      = pinn_manifest['feature_cols']
target_cols       = pinn_manifest['target_cols']
state_target_cols = pinn_manifest.get('state_target_cols', [])
species_cols      = pinn_manifest.get('species_cols', [])

pinn_scalers = joblib.load(MODELS_DIR / 'pinn_pfr' / 'pinn_pfr_scalers.joblib')
scaler_X = pinn_scalers['scaler_X']
scaler_y = pinn_scalers.get('scaler_y')

pinn_model = PINNPFR(
    arch['in_features'], arch['h1'], arch['h2'], arch['h3'],
    arch['out_features'], dropout=arch['dropout'],
)
pinn_model.load_state_dict(torch.load(
    MODELS_DIR / 'pinn_pfr' / 'pinn_pfr_state_dict.pt', map_location='cpu', weights_only=True,
))
pinn_model.eval()

print(f'PINN loaded: {sum(p.numel() for p in pinn_model.parameters()):,} params')
print(f'Inputs  ({len(feature_cols)}): {feature_cols}')
print(f'Outputs ({len(target_cols)}): {target_cols}')

In [ ]:
# ═════════════════════════════════════════════════════════════════════════
# 4. LOAD SR EQUATIONS (Main_8)
# ═════════════════════════════════════════════════════════════════════════
sr_eq_path       = SR_DIR / f'{SR_SUBDIR}_equations.py'
sr_manifest_path = SR_DIR / f'{SR_SUBDIR}_manifest.json'
if not sr_eq_path.exists():
    raise FileNotFoundError(
        f'{sr_eq_path} not found — run Main_8 with TEACHER_STEM={SR_TEACHER_STEM!r} '
        f'and IF_SR_EXPORT=True first.'
    )

spec = importlib.util.spec_from_file_location(f'{SR_SUBDIR}_equations', sr_eq_path)
sr_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sr_module)

sr_manifest = json.loads(sr_manifest_path.read_text()) if sr_manifest_path.exists() else {}
sr_inlet_cols = sr_manifest.get('inlet_cols') or feature_cols


def sr_predict(target, df):
    """Vectorised call into the PySR-exported function for one target."""
    fn_name = 'predict_' + target.replace('.', '_').replace(' ', '_')
    fn = getattr(sr_module, fn_name, None)
    if fn is None:
        return None
    args = [df[c].to_numpy(dtype=float) for c in sr_inlet_cols]
    return np.asarray(fn(*args), dtype=float)


sr_available_targets = [
    t for t in target_cols
    if hasattr(sr_module, 'predict_' + t.replace('.', '_').replace(' ', '_'))
]
print(f'SR equations loaded: {len(sr_available_targets)}/{len(target_cols)} targets available')

In [ ]:
# ═════════════════════════════════════════════════════════════════════════
# 5. LOAD DATA & SELECT COMPARISON RUNS
# ═════════════════════════════════════════════════════════════════════════
pkl_files = sorted((project_root / 'data' / 'processed').glob('features_targets_*.pkl'))
if not pkl_files:
    raise FileNotFoundError('No features_targets_*.pkl in data/processed. Run Main_3.')
loaded = load_portable_pickle(str(pkl_files[-1]))
df_features = loaded['df_features']
df_target   = loaded['df_target']
print(f'Loaded {len(df_features):,} rows from {pkl_files[-1].name}')

avail_feature_cols = [c for c in feature_cols if c in df_features.columns]
avail_target_cols  = [c for c in target_cols if c in df_target.columns]
if len(avail_feature_cols) < len(feature_cols):
    raise KeyError(f'Feature columns missing from processed data: {set(feature_cols) - set(avail_feature_cols)}')

run_cols = [c for c in avail_feature_cols if c not in ('z_position_m', 'relative_position')]
run_id = df_features.groupby(run_cols, dropna=False).ngroup()
unique_runs = np.array(sorted(pd.unique(run_id)))

rng = np.random.RandomState(RANDOM_STATE)
comparison_runs = rng.choice(unique_runs, size=min(N_COMPARISON_RUNS, len(unique_runs)), replace=False)
comparison_mask = run_id.isin(comparison_runs)

print(f'Total runs: {len(unique_runs)}  |  Comparison runs sampled: {len(comparison_runs)}')

In [ ]:
# ═════════════════════════════════════════════════════════════════════════
# 6. COMPUTE PREDICTIONS (CANTERA GROUND TRUTH ALREADY IN df_target)
# ═════════════════════════════════════════════════════════════════════════
X_all = df_features[avail_feature_cols].copy()
y_cantera = df_target[avail_target_cols].copy()

X_scaled = scaler_X.transform(X_all.values)
with torch.no_grad():
    y_pinn_scaled = pinn_model(torch.tensor(X_scaled, dtype=torch.float32)).numpy()
y_pinn = scaler_y.inverse_transform(y_pinn_scaled) if scaler_y is not None else y_pinn_scaled
df_pinn = pd.DataFrame(y_pinn, columns=target_cols, index=X_all.index)[avail_target_cols]

sr_columns = {}
for t in avail_target_cols:
    pred = sr_predict(t, X_all)
    if pred is not None:
        sr_columns[t] = pred
df_sr = pd.DataFrame(sr_columns, index=X_all.index)
sr_covered_targets = list(df_sr.columns)

print(f'Predictions computed for {len(X_all):,} rows.')
print(f'  PINN targets: {len(avail_target_cols)}')
print(f'  SR targets covered: {len(sr_covered_targets)}/{len(avail_target_cols)}')

In [ ]:
# ═════════════════════════════════════════════════════════════════════════
# 7. AXIAL PROFILE OVERLAYS — CANTERA vs PINN vs SR
# ═════════════════════════════════════════════════════════════════════════
_preferred_state = [c for c in ['temperature_K', 'pressure_Pa', 'velocity_ms', 'density_kgm3'] if c in avail_target_cols]
_top_species = sorted(species_cols)[:2] if species_cols else []
overlay_targets = (_preferred_state + _top_species) or avail_target_cols[:4]
overlay_targets = [t for t in overlay_targets if t in avail_target_cols]

z_col = 'relative_position' if 'relative_position' in X_all.columns else avail_feature_cols[-1]

for run in comparison_runs[: min(3, len(comparison_runs))]:
    _mask = (run_id == run).to_numpy()
    _order = np.argsort(X_all.loc[_mask, z_col].to_numpy())
    z = X_all.loc[_mask, z_col].to_numpy()[_order]

    n_cols = min(3, len(overlay_targets))
    n_rows = int(np.ceil(len(overlay_targets) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.5 * n_cols, 3.6 * n_rows), squeeze=False)
    setup_matplotlib(axes)

    for i, t in enumerate(overlay_targets):
        ax = axes[i // n_cols][i % n_cols]
        ax.plot(z, y_cantera.loc[_mask, t].to_numpy()[_order], color='k', lw=1.8, label='Cantera')
        ax.plot(z, df_pinn.loc[_mask, t].to_numpy()[_order], color='b', lw=1.4, ls='--', label='PINN')
        if t in sr_covered_targets:
            ax.plot(z, df_sr.loc[_mask, t].to_numpy()[_order], color='r', lw=1.4, ls='-.', label='SR')
        ax.set_xlabel('z/L')
        ax.set_title(t, fontsize=9)
        if i == 0:
            ax.legend(fontsize=8)

    for j in range(len(overlay_targets), n_rows * n_cols):
        axes[j // n_cols][j % n_cols].set_visible(False)

    fig.suptitle(f'Axial profile — Cantera vs PINN vs SR (run_id={run})', y=1.02)
    plt.tight_layout()
    if IF_PLOT_EXPORT:
        fig.savefig(FIGS_DIR / f'axial_overlay_run_{run}.png', dpi=180, bbox_inches='tight')
    if IF_PLOT_SHOWN:
        plt.show()
    plt.close(fig)

In [ ]:
# ═════════════════════════════════════════════════════════════════════════
# 8. PARITY PLOTS — PINN vs CANTERA, SR vs CANTERA
# ═════════════════════════════════════════════════════════════════════════
parity_targets = overlay_targets
n_cols = len(parity_targets)
fig, axes = plt.subplots(2, n_cols, figsize=(4.2 * n_cols, 7.6), squeeze=False)
setup_matplotlib(axes)

for i, t in enumerate(parity_targets):
    yt = y_cantera[t].to_numpy()

    ax = axes[0][i]
    yp = df_pinn[t].to_numpy()
    ax.scatter(yt, yp, s=5, alpha=0.25, c='b', rasterized=True)
    lims = [min(yt.min(), yp.min()), max(yt.max(), yp.max())]
    ax.plot(lims, lims, 'r-', lw=1)
    r2 = r2_score(yt, yp)
    ax.set_title(f'PINN — {t}\nR²={r2:.4f}', fontsize=9)
    ax.set_xlabel('Cantera')
    ax.set_ylabel('PINN')

    ax = axes[1][i]
    if t in sr_covered_targets:
        yp = df_sr[t].to_numpy()
        ax.scatter(yt, yp, s=5, alpha=0.25, c='r', rasterized=True)
        lims = [min(yt.min(), yp.min()), max(yt.max(), yp.max())]
        ax.plot(lims, lims, 'k-', lw=1)
        r2 = r2_score(yt, yp)
        ax.set_title(f'SR — {t}\nR²={r2:.4f}', fontsize=9)
    else:
        ax.set_title(f'SR — {t}\n(not covered)', fontsize=9)
    ax.set_xlabel('Cantera')
    ax.set_ylabel('SR')

fig.suptitle('Parity: surrogate vs Cantera ground truth (all rows)', y=1.02)
plt.tight_layout()
if IF_PLOT_EXPORT:
    fig.savefig(FIGS_DIR / 'parity_pinn_sr_vs_cantera.png', dpi=180, bbox_inches='tight')
if IF_PLOT_SHOWN:
    plt.show()
plt.close(fig)

In [ ]:
# ═════════════════════════════════════════════════════════════════════════
# 9. METRICS TABLE — R², MAE, NMAE PER TARGET
# ═════════════════════════════════════════════════════════════════════════
def _nmae(yt, yp):
    mae = mean_absolute_error(yt, yp)
    yrange = np.max(yt) - np.min(yt)
    if yrange > 1e-12:
        return mae / yrange * 100.0
    ymean = np.mean(np.abs(yt))
    return mae / ymean * 100.0 if ymean > 1e-12 else 0.0

rows = []
for t in avail_target_cols:
    yt = y_cantera[t].to_numpy()
    yp_pinn = df_pinn[t].to_numpy()
    row = {
        'target': t,
        'R2_pinn': r2_score(yt, yp_pinn),
        'NMAE_pinn_%': _nmae(yt, yp_pinn),
    }
    if t in sr_covered_targets:
        yp_sr = df_sr[t].to_numpy()
        row['R2_sr'] = r2_score(yt, yp_sr)
        row['NMAE_sr_%'] = _nmae(yt, yp_sr)
    else:
        row['R2_sr'] = np.nan
        row['NMAE_sr_%'] = np.nan
    rows.append(row)

df_comparison_metrics = pd.DataFrame(rows)
print(df_comparison_metrics.to_string(index=False))
print(f"\nMean R²  — PINN: {df_comparison_metrics.R2_pinn.mean():.4f}  |  SR: {df_comparison_metrics.R2_sr.mean():.4f}")
print(f"Mean NMAE — PINN: {df_comparison_metrics['NMAE_pinn_%'].mean():.2f}%  |  SR: {df_comparison_metrics['NMAE_sr_%'].mean():.2f}%")

fig, ax = plt.subplots(figsize=(max(8, 0.5 * len(avail_target_cols)), 4.5))
setup_matplotlib(ax)
x = np.arange(len(df_comparison_metrics))
width = 0.35
ax.bar(x - width / 2, df_comparison_metrics['R2_pinn'], width, facecolor='white', edgecolor='b', hatch='///', label='PINN')
ax.bar(x + width / 2, df_comparison_metrics['R2_sr'], width, facecolor='white', edgecolor='r', hatch='', label='SR')
ax.axhline(1.0, color='k', lw=0.8, ls='--')
ax.axhline(0.0, color='k', lw=0.8, ls=':')
ax.set_xticks(x)
ax.set_xticklabels(df_comparison_metrics['target'], rotation=90, fontsize=7)
ax.set_ylabel('R² vs Cantera')
ax.set_title('Per-target fidelity: PINN vs SR (both vs Cantera)')
ax.legend(fontsize=8)
plt.tight_layout()
if IF_PLOT_EXPORT:
    fig.savefig(FIGS_DIR / 'r2_bar_pinn_vs_sr.png', dpi=180, bbox_inches='tight')
if IF_PLOT_SHOWN:
    plt.show()
plt.close(fig)

In [ ]:
# ═════════════════════════════════════════════════════════════════════════
# 10. INFERENCE SPEED — PINN vs SR
# ═════════════════════════════════════════════════════════════════════════
_n_speed = min(2000, len(X_all))
X_speed = X_all.iloc[:_n_speed]
X_speed_scaled = scaler_X.transform(X_speed.values)

with torch.no_grad():
    _ = pinn_model(torch.tensor(X_speed_scaled[:10], dtype=torch.float32))  # warm-up
start = time.perf_counter()
with torch.no_grad():
    _ = pinn_model(torch.tensor(X_speed_scaled, dtype=torch.float32))
pinn_seconds = time.perf_counter() - start

start = time.perf_counter()
for t in sr_covered_targets:
    sr_predict(t, X_speed)
sr_seconds = time.perf_counter() - start

print(f'PINN: {_n_speed:,} rows in {pinn_seconds * 1e3:.2f} ms  ({_n_speed / pinn_seconds:,.0f} rows/s)')
print(f'SR:   {_n_speed:,} rows in {sr_seconds * 1e3:.2f} ms  ({_n_speed / sr_seconds:,.0f} rows/s)  '
      f'({len(sr_covered_targets)} targets)')
if sr_seconds > 0:
    print(f'SR speedup vs PINN: {pinn_seconds / sr_seconds:.1f}x')

In [ ]:
# ═════════════════════════════════════════════════════════════════════════
# 11. EXPORT COMPARISON REPORT
# ═════════════════════════════════════════════════════════════════════════
if IF_EXPORT_REPORT:
    metrics_csv = FIGS_DIR / 'comparison_metrics.csv'
    df_comparison_metrics.to_csv(metrics_csv, index=False)

    comparison_manifest = {
        'run_at': datetime.now().isoformat(timespec='seconds'),
        'pinn_manifest': str(pinn_manifest_path),
        'sr_teacher_stem': SR_TEACHER_STEM,
        'sr_manifest': str(sr_manifest_path) if sr_manifest_path.exists() else None,
        'n_rows_compared': int(len(X_all)),
        'n_comparison_runs': int(len(comparison_runs)),
        'targets_compared': avail_target_cols,
        'sr_covered_targets': sr_covered_targets,
        'mean_r2_pinn': float(df_comparison_metrics['R2_pinn'].mean()),
        'mean_r2_sr': float(df_comparison_metrics['R2_sr'].mean()) if sr_covered_targets else None,
        'pinn_inference_seconds': pinn_seconds,
        'sr_inference_seconds': sr_seconds,
        'metrics_csv': str(metrics_csv),
    }
    comparison_manifest_path = FIGS_DIR / 'comparison_manifest.json'
    with open(comparison_manifest_path, 'w') as f:
        json.dump(comparison_manifest, f, indent=2)
    print(f'[OK] metrics  -> {metrics_csv}')
    print(f'[OK] manifest -> {comparison_manifest_path}')
else:
    print('[SKIP] IF_EXPORT_REPORT=False — no files written.')

---

## Summary

- **Cantera**: ground truth, from the processed `features_targets_*.pkl` dataset (Main_3).
- **PINN** (Main_7): physics-informed neural network, full axial profile.
- **SR** (Main_8): closed-form symbolic regression distilled from the PINN teacher (`TEACHER_STEM='pinn_pfr'`).
- Axial profile overlays and parity plots show where SR tracks (or diverges from) its PINN teacher, and where both track (or diverge from) Cantera.
- Inference speed: SR is typically much faster than PINN (pure NumPy vs a PyTorch forward pass) — relevant when picking a surrogate for downstream optimisation (Main_10) or process-simulator embedding.

Use `Main_10_optimisation_BO_surrogate_vs_cantera.ipynb` to optimise inlet conditions using the validated SR surrogate.